# Bayesian Hypothesis Testing Approaches
## Full Extended Project — Solution Notebook

**Goal.** Move from classical p-value hypothesis testing to Bayesian approaches that answer the questions decision-makers actually ask:

- “What is the probability that the Vein Pack mean lifespan exceeds 71 years?”
- “What is the probability that Vein is better than Artery?”
- “How sensitive is that conclusion to my prior beliefs?”

We cover conjugate updates (Beta-Binomial for rates, Normal/t for means), posterior decision metrics, ROPE thinking, Bayes-factor intuition, alternate implementations, extra practice, and a simulation section that lets you change prior strength and sample size.

**Data used**
- Familiar lifespan CSV (`data/familiar_lifespan.csv`) — same data as the frequentist R project
- Synthetic A/B conversion counts for the classic Beta-Binomial case

**Audience note (from the supplied PDFs).** Bayesian output is naturally expressed as probabilities (“there is a 97 % chance that…”). This language is usually far more accessible to product managers and executives than p-values, while still giving technical reviewers the full posterior distribution and sensitivity analysis.


## Flowchart — Bayesian Hypothesis-Testing Workflow

![Bayesian HT Flowchart](bayesian_ht_flowchart.png)

The flowchart emphasises the continuous flow Prior → Likelihood → Posterior → Decision quantities expressed as probabilities, rather than a binary “reject / fail to reject” at a fixed α.


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

# Reproducibility
rng = np.random.default_rng(42)

# Familiar data
lifespans = pd.read_csv("data/familiar_lifespan.csv")
vein    = lifespans.loc[lifespans.pack == "vein",    "lifespan"].values
artery  = lifespans.loc[lifespans.pack == "artery",  "lifespan"].values

print("Vein  : n =", len(vein),   " mean =", round(vein.mean(), 3),   " sd =", round(vein.std(ddof=1), 3))
print("Artery: n =", len(artery), " mean =", round(artery.mean(), 3), " sd =", round(artery.std(ddof=1), 3))


## 1. Frequentist vs Bayesian mindset (quick contrast)

| Question the decision-maker asks | Frequentist answer | Bayesian answer |
|----------------------------------|--------------------|-----------------|
| Is Vein better than 71? | p-value = 2.7 × 10⁻¹⁰ → reject H₀ | P(μ_vein > 71 | data) ≈ 1.00 |
| Is Vein better than Artery? | p ≈ 0.056 → fail to reject | P(μ_vein > μ_artery | data) ≈ 0.97 |
| How sure are we? | Confidence interval (random interval that covers the true value 95 % of the time) | Credible interval / HDI (interval that contains 95 % of the posterior probability) |
| Role of prior information | Not used (or used only in design) | Explicitly encoded and updated |

Bayesian methods do **not** eliminate the need for careful design or for checking assumptions; they change the language of the final claim from “we reject the null” to “the posterior probability of the claim is X”.


## 2. Beta-Binomial model — Bayesian A/B testing for conversion rates

Classic conjugate example.  
Prior: θ ~ Beta(α₀, β₀)  
Likelihood: k conversions out of n trials ~ Binomial(n, θ)  
Posterior: θ | data ~ Beta(α₀ + k, β₀ + n − k)

### 2.1 Observed data (synthetic but realistic)


In [ ]:
# Landing-page experiment
control_visitors     = 1200
control_conversions  = 120          # 10.0 %
treatment_visitors   = 1200
treatment_conversions = 156         # 13.0 %

print(f"Control   rate = {control_conversions / control_visitors:.1%}")
print(f"Treatment rate = {treatment_conversions / treatment_visitors:.1%}")


### 2.2 Choose a weakly informative prior and update

In [ ]:
# Uniform / Jeffreys-style weak prior
prior_a, prior_b = 1, 1

# Posterior parameters
post_c_a = prior_a + control_conversions
post_c_b = prior_b + (control_visitors - control_conversions)
post_t_a = prior_a + treatment_conversions
post_t_b = prior_b + (treatment_visitors - treatment_conversions)

print(f"Control   posterior: Beta({post_c_a}, {post_c_b})")
print(f"Treatment posterior: Beta({post_t_a}, {post_t_b})")
print(f"Posterior means: control = {post_c_a/(post_c_a+post_c_b):.4f}, treatment = {post_t_a/(post_t_a+post_t_b):.4f}")


### 2.3 Decision metric via Monte-Carlo

In [ ]:
n_mc = 150_000
samples_c = stats.beta.rvs(post_c_a, post_c_b, size=n_mc, random_state=1)
samples_t = stats.beta.rvs(post_t_a, post_t_b, size=n_mc, random_state=2)

prob_treat_better = (samples_t > samples_c).mean()
expected_lift     = (samples_t - samples_c).mean()
prob_lift_gt_1pp  = ((samples_t - samples_c) > 0.01).mean()

print(f"P(treatment > control | data)          = {prob_treat_better:.4f}")
print(f"Expected lift (absolute)               = {expected_lift:.4f}")
print(f"P(lift > 1 percentage point | data)    = {prob_lift_gt_1pp:.4f}")


## 3. Bayesian comparison of means — Familiar Vein Pack

We place a flat (non-informative) prior on the mean.  
Under a normal likelihood the posterior for μ is a scaled-and-shifted Student-t distribution with n−1 degrees of freedom.  
We draw Monte-Carlo samples from that posterior and answer probability questions directly.


In [ ]:
def posterior_mean_samples(x, n_samples=200_000, rng=None):
    # Posterior samples of mu under a flat prior (Normal likelihood)
    n    = len(x)
    xbar = np.mean(x)
    s    = np.std(x, ddof=1)
    t    = rng.standard_t(df=n-1, size=n_samples)
    return xbar + (s / np.sqrt(n)) * t

post_vein   = posterior_mean_samples(vein,   rng=rng)
post_artery = posterior_mean_samples(artery, rng=rng)

# Key decision probabilities
p_gt_71     = (post_vein > 71).mean()
p_vein_gt_a = (post_vein > post_artery).mean()

print(f"P(mu_Vein > 71          | data) ≈ {p_gt_71:.6f}")
print(f"P(mu_Vein > mu_Artery    | data) ≈ {p_vein_gt_a:.4f}")


### 3.1 Highest Density Interval (credible interval)

In [ ]:
def hdi(samples, cred_mass=0.95):
    samples = np.sort(samples)
    n = len(samples)
    interval_size = int(np.floor(cred_mass * n))
    widths = samples[interval_size:] - samples[:n - interval_size]
    idx = np.argmin(widths)
    return samples[idx], samples[idx + interval_size]

vein_hdi = hdi(post_vein)
diff_hdi = hdi(post_vein - post_artery)
print(f"Vein mean 95% HDI          : [{vein_hdi[0]:.2f}, {vein_hdi[1]:.2f}] years")
print(f"(Vein - Artery) 95% HDI    : [{diff_hdi[0]:.2f}, {diff_hdi[1]:.2f}] years")


### 3.2 Visualise the posteriors

![Posterior distributions](bayesian_ht_posteriors.png)


## 4. ROPE and practical equivalence

Instead of testing a point null, many Bayesian practitioners define a **Region of Practical Equivalence (ROPE)** — an interval around zero (or around the null value) that is considered “practically the same”.

Example: if a difference of less than 0.5 years is clinically unimportant for Familiar, we can ask:

- What is the posterior probability that the true difference lies inside the ROPE [−0.5, +0.5]?
- What is the probability that it lies above the ROPE (clear superiority)?


In [ ]:
rope_low, rope_high = -0.5, 0.5
diff = post_vein - post_artery

p_in_rope   = ((diff >= rope_low) & (diff <= rope_high)).mean()
p_above     = (diff > rope_high).mean()
p_below     = (diff < rope_low).mean()

print(f"P(difference in ROPE [{rope_low}, {rope_high}]) = {p_in_rope:.3f}")
print(f"P(Vein clearly better  > {rope_high})          = {p_above:.3f}")
print(f"P(Artery clearly better < {rope_low})         = {p_below:.3f}")


## 5. Alternate code paths

### 5.1 Analytic Beta quantiles (no Monte-Carlo needed for simple questions)


In [ ]:
# 95% central credible interval for treatment conversion rate
lo, hi = stats.beta.ppf([0.025, 0.975], post_t_a, post_t_b)
print(f"Treatment rate 95% central CI: [{lo:.4f}, {hi:.4f}]")

# Probability treatment rate > 0.12 (analytic)
p_gt_12 = 1 - stats.beta.cdf(0.12, post_t_a, post_t_b)
print(f"P(theta_treat > 0.12 | data) = {p_gt_12:.4f}")


### 5.2 Stronger / weaker priors (sensitivity)

In [ ]:
def beta_ab_prob(prior_a, prior_b, k_c, n_c, k_t, n_t, n_mc=80000):
    pa = prior_a + k_c; pb = prior_b + (n_c - k_c)
    qa = prior_a + k_t; qb = prior_b + (n_t - k_t)
    sc = stats.beta.rvs(pa, pb, size=n_mc, random_state=10)
    st = stats.beta.rvs(qa, qb, size=n_mc, random_state=11)
    return (st > sc).mean()

print("P(T>C) under different priors:")
for a,b,label in [(1,1,"Uniform"), (5,5,"Weakly informative"), (50,50,"Strongly skeptical")]:
    p = beta_ab_prob(a, b, control_conversions, control_visitors,
                     treatment_conversions, treatment_visitors)
    print(f"  Beta({a},{b})  ->  {p:.4f}   ({label})")


### 5.3 Bootstrap-style posterior (non-parametric alternative)

In [ ]:
# Resample the observed lifespans many times -> approximate posterior of the mean
boot_vein   = np.array([rng.choice(vein,   size=len(vein),   replace=True).mean() for _ in range(20000)])
boot_artery = np.array([rng.choice(artery, size=len(artery), replace=True).mean() for _ in range(20000)])
print(f"Bootstrap P(mu_Vein > mu_Artery) ≈ {(boot_vein > boot_artery).mean():.4f}")


## 6. More Practice

1. Change the A/B counts to a smaller experiment (e.g. 300 visitors each) and recompute P(T > C). How much does the probability drop?
2. For the Familiar data, compute P(μ_Vein > 75) and P(μ_Vein > 78).
3. Define a ROPE of ±1 year around zero for the pack difference and recalculate the three probabilities.
4. Using the analytic Beta CDF, find the smallest conversion rate threshold τ such that P(θ_treat > τ | data) ≥ 0.95.


In [ ]:
# Practice answers (uncomment to check)
print("P(mu_Vein > 75) =", (post_vein > 75).mean())
print("P(mu_Vein > 78) =", (post_vein > 78).mean())


## 7. Simulation Section — Prior strength & sample-size effects

We can vary two key inputs and watch how the posterior probability changes:

- Strength of the prior (α₀ = β₀)
- Number of observed visitors (or subscribers)


In [ ]:
def simulate_power_like(n_visitors, true_p_c=0.10, true_p_t=0.13,
                        prior_a=1, prior_b=1, n_sims=2000, n_mc=5000):
    # Fraction of simulated experiments in which P(T>C | data) > 0.95
    wins = 0
    for i in range(n_sims):
        k_c = rng.binomial(n_visitors, true_p_c)
        k_t = rng.binomial(n_visitors, true_p_t)
        pa, pb = prior_a + k_c, prior_b + (n_visitors - k_c)
        qa, qb = prior_a + k_t, prior_b + (n_visitors - k_t)
        sc = stats.beta.rvs(pa, pb, size=n_mc)
        st = stats.beta.rvs(qa, qb, size=n_mc)
        if (st > sc).mean() > 0.95:
            wins += 1
    return wins / n_sims

# Effect of sample size under a fixed weak prior
ns = [200, 400, 800, 1200, 2000]
print("Approx. 'power' (P(decision > 0.95) when true lift exists):")
for n in ns:
    pwr = simulate_power_like(n, n_sims=600)
    print(f"  n = {n:4d} per arm  ->  {pwr:.3f}")


In [ ]:
# Sensitivity of the Familiar posterior probability to a Normal prior mean
# (illustrative: pull the prior mean of the difference toward 0 with increasing strength)
def normal_posterior_prob(prior_mean=0.0, prior_sd=10.0, n_samples=50000):
    # Approximate: likelihood N(diff_hat, se), prior N(prior_mean, prior_sd)
    diff_hat = vein.mean() - artery.mean()
    se = np.sqrt(vein.var(ddof=1)/len(vein) + artery.var(ddof=1)/len(artery))
    # Precision-weighted posterior
    post_prec = 1/prior_sd**2 + 1/se**2
    post_mean = (prior_mean/prior_sd**2 + diff_hat/se**2) / post_prec
    post_sd   = np.sqrt(1/post_prec)
    samples = rng.normal(post_mean, post_sd, size=n_samples)
    return (samples > 0).mean()

print("P(diff > 0) under Normal priors centred at 0 with different strengths:")
for sd in [20, 5, 2, 1, 0.5]:
    p = normal_posterior_prob(prior_mean=0.0, prior_sd=sd)
    print(f"  prior SD = {sd:4.1f}  ->  P(diff>0) = {p:.4f}")


### Interactive parameter block (edit & re-run)

```python
# ---- EDIT THESE ----
MY_N_VISITORS   = 800
MY_PRIOR_A      = 2
MY_PRIOR_B      = 2
MY_TRUE_LIFT    = 0.03          # treatment - control
# --------------------
print(simulate_power_like(MY_N_VISITORS, true_p_t=0.10+MY_TRUE_LIFT,
                          prior_a=MY_PRIOR_A, prior_b=MY_PRIOR_B, n_sims=600))
```


## 8. Key Takeaways & Familiar Recommendations

1. **Vein Pack vs 71-year benchmark**  
   Posterior probability that the true mean exceeds 71 is essentially 1.  
   Bayesian language: “We are virtually certain the Vein Pack extends life beyond the population average.”

2. **Vein vs Artery**  
   P(μ_Vein > μ_Artery | data) ≈ 0.97.  
   The 95 % HDI for the difference still includes a small negative value, consistent with the frequentist p ≈ 0.056.  
   A ROPE analysis shows most posterior mass lies above a ±0.5-year equivalence region.

3. **Communication advantage**  
   Product and executive audiences understand “97 % probability that Vein is better” far more readily than “p = 0.056, fail to reject the null.”  
   Technical reviewers still receive the full posterior, HDI, and sensitivity-to-prior results.

4. **Design implication**  
   The simulation section shows how many more observations are needed before a Bayesian decision threshold of 0.95 (or 0.99) is reached with high reliability under a realistic lift.

5. **When to prefer Bayesian**  
   Continuous monitoring, need for direct probability statements, incorporation of historical / domain knowledge, and multi-arm or sequential settings.


## Appendix — Reproducibility

In [ ]:
import sys, platform
print(platform.python_version())
print("numpy", np.__version__)
